# Effective Tiers Export

Standalone script that computes `effective_tiers` per `(product_id, warehouse_id)` and exports to Excel + Snowflake.

## Pipeline
1. Fetch V2 price tiers from `get_market_data_v2()` (competitor/market prices per product × region)
2. Fetch margin tiers from `get_margin_tiers()` (historical margins per product × warehouse, with cascade fallback)
3. Fetch WAC from Snowflake
4. Expand V2 to warehouse level, convert margin tiers to prices
5. Build `effective_tiers` = `price_tiers` (V2) > `margin_tier_prices` > empty
6. Export to Excel + upload to Snowflake

## Usage
Run all cells. Output: `effective_tiers_export.xlsx` + Snowflake table `MATERIALIZED_VIEWS.effective_tiers_export`.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime
import pytz

sys.path.insert(0, os.path.abspath('..'))
from constants import WAREHOUSE_MAPPING, COHORT_IDS

# Load market_data_module_2 (which internally loads V1 + queries_module)
_md2_path = 'market_data_module_2.ipynb' if os.path.exists('market_data_module_2.ipynb') else 'modules/market_data_module_2.ipynb'
%run $_md2_path

from common_functions import upload_dataframe_to_snowflake

CAIRO_TZ = pytz.timezone('Africa/Cairo')

# Build cohort → warehouse list mapping from constants
COHORT_WAREHOUSES = {}
for region, wh_name, wh_id, cohort_id in WAREHOUSE_MAPPING:
    COHORT_WAREHOUSES.setdefault(cohort_id, []).append(wh_id)

print(f'Effective Tiers Export ready at {datetime.now(CAIRO_TZ).strftime("%Y-%m-%d %H:%M:%S")} Cairo time')
print(f'Cohort → warehouse mapping: {COHORT_WAREHOUSES}')

In [ ]:
# =============================================================================
# FETCH DATA
# =============================================================================
print('=' * 60)
print('EFFECTIVE TIERS EXPORT')
print('=' * 60)

# 1. V2 price tiers per (product_id, region)
print('\n1. Fetching V2 price tiers...')
df_v2, _ = get_market_data_v2()
print(f'   {len(df_v2)} product-region rows with price tiers')

# 2. Margin tiers per (product_id, warehouse_id) with cascade fallback
print('\n2. Fetching margin tiers...')
df_tiers = get_margin_tiers()
print(f'   {len(df_tiers)} product-warehouse rows with margin tiers')

# 3. WAC per product
print('\n3. Fetching WAC...')
df_wac = query_snowflake(V2_WAC_QUERY)
print(f'   {len(df_wac)} products with WAC')

In [ ]:
# =============================================================================
# BUILD EFFECTIVE TIERS
# =============================================================================

# --- Step A: Expand V2 price_tiers to warehouse level ---
print('\n4. Expanding V2 price tiers to warehouse level...')
df_v2_cohorts = expand_to_cohorts(df_v2)

v2_wh_rows = []
for _, row in df_v2_cohorts.iterrows():
    cohort = row['cohort_id']
    for wh_id in COHORT_WAREHOUSES.get(cohort, []):
        v2_wh_rows.append({
            'product_id': row['product_id'],
            'warehouse_id': wh_id,
            'price_tiers': row['price_tiers'],
        })

df_v2_wh = pd.DataFrame(v2_wh_rows)
df_v2_wh = df_v2_wh.drop_duplicates(subset=['product_id', 'warehouse_id'])
print(f'   {len(df_v2_wh)} product-warehouse rows with V2 price tiers')

# --- Step B: Convert margin tiers to prices ---
print('\n5. Converting margin tiers to prices...')
df_mt = df_tiers.merge(df_wac, on='product_id', how='left')

MARGIN_TIER_COLS = [
    'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
    'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2'
]

def build_margin_tier_prices(row):
    wac = row.get('wac_p', 0)
    if not wac or wac <= 0:
        return []
    prices = []
    for col in MARGIN_TIER_COLS:
        m = row.get(col)
        if pd.notna(m) and 0 < m < 1:
            prices.append(round(wac / (1 - m) * 4) / 4)
    return sorted(set(prices))

df_mt['margin_tier_prices'] = df_mt.apply(build_margin_tier_prices, axis=1)
has_mt = (df_mt['margin_tier_prices'].apply(len) > 0).sum()
print(f'   {has_mt} product-warehouse rows with valid margin tier prices')

# --- Step C: Merge and build effective_tiers ---
print('\n6. Building effective tiers...')
df_out = df_mt[['product_id', 'warehouse_id', 'margin_tier_prices']].merge(
    df_v2_wh[['product_id', 'warehouse_id', 'price_tiers']],
    on=['product_id', 'warehouse_id'],
    how='outer'
)
df_out['price_tiers'] = df_out['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])
df_out['margin_tier_prices'] = df_out['margin_tier_prices'].apply(lambda x: x if isinstance(x, list) else [])

def get_effective_tiers(row):
    if row['price_tiers'] and len(row['price_tiers']) > 0:
        return row['price_tiers']
    if row['margin_tier_prices'] and len(row['margin_tier_prices']) > 0:
        return row['margin_tier_prices']
    return []

df_out['effective_tiers'] = df_out.apply(get_effective_tiers, axis=1)

# Final rounding + dedup
df_out['effective_tiers'] = df_out['effective_tiers'].apply(
    lambda tiers: sorted(set(round(p * 4) / 4 for p in tiers))
)

# Track source
def get_tier_source(row):
    if row['price_tiers'] and len(row['price_tiers']) > 0:
        return 'price_tiers'
    if row['margin_tier_prices'] and len(row['margin_tier_prices']) > 0:
        return 'margin_tier_prices'
    return 'none'

df_out['tier_source'] = df_out.apply(get_tier_source, axis=1)

print(f'\n   Total rows: {len(df_out)}')
print(f'   With effective tiers: {(df_out["effective_tiers"].apply(len) > 0).sum()}')
print(f'   Empty effective tiers: {(df_out["effective_tiers"].apply(len) == 0).sum()}')
print(f'   Source distribution:')
print(f'     {df_out["tier_source"].value_counts().to_dict()}')
print(f'   Avg tiers per SKU: {df_out["effective_tiers"].apply(len).mean():.1f}')

In [ ]:
# =============================================================================
# EXPORT TO EXCEL + SNOWFLAKE
# =============================================================================
print('\n7. Exporting...')

df_export = df_out[['product_id', 'warehouse_id', 'effective_tiers', 'tier_source']].copy()
df_export['effective_tiers'] = df_export['effective_tiers'].apply(str)
df_export['num_tiers'] = df_out['effective_tiers'].apply(len)
df_export['created_at'] = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')

# Excel
excel_path = 'effective_tiers_export.xlsx'
df_export.to_excel(excel_path, index=False)
print(f'   Saved to {excel_path} ({len(df_export)} rows)')

# Snowflake
print('   Uploading to Snowflake...')
upload_status = upload_dataframe_to_snowflake(
    "Egypt",
    df_export,
    "MATERIALIZED_VIEWS",
    "effective_tiers_export",
    "overwrite",
    auto_create_table=True
)
print(f'   Snowflake upload: {upload_status}')

print(f'\n{"=" * 60}')
print('EFFECTIVE TIERS EXPORT COMPLETE')
print(f'{"=" * 60}')
print(f'Total: {len(df_export)} product-warehouse rows')
print(f'With tiers: {(df_export["num_tiers"] > 0).sum()}')
print(f'Empty: {(df_export["num_tiers"] == 0).sum()}')